# Modelo Espacio-Temporal: CNN-LSTM (Modelo 3)

Este cuaderno implementa el entrenamiento y la evaluación de la arquitectura combinada **CNN-LSTM** para la estimación del tiempo transcurrido desde el contacto del rastro térmico.

### Características de la arquitectura:
1. **Backbone Visual (CNN 2D):** Utiliza la rama ligera **Lite-DSTFS** (~750K parámetros) heredada de MTDE-Net para extraer características espaciales del decaimiento térmico cuadro por cuadro.
2. **Capa Recurrente (LSTM):** Modela la dinámica de enfriamiento temporal sobre secuencias de características visuales extraídas por la CNN.
3. **Ventanas Móviles (Sliding Windows):** Genera ventanas deslizantes de tamaño fijo (por defecto, $L=5$ fotogramas consecutivos) garantizando la consistencia física de que todos los cuadros dentro de una ventana pertenezcan a la misma secuencia de contacto (`sequence_id`).

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.models.cnn_lstm import ThermalCNNLSTM
from src.loaders.cnn_lstm_loader import ThermalSequenceDataset
from src.utils import SqrtScaledMSELoss, eval_cnn_lstm_metrics

### 1. Semilla Global y Reproducibilidad

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### 2. Configuración General (Hiperparámetros)

In [3]:
CONFIG = {
    "epochs": 80,
    "patience": 12,
    "min_delta": 1.0,
    "batch_size": 8,           # Un batch size menor es recomendado para datos secuenciales pesados
    "lr": 0.0003,              # AdamW tasa de aprendizaje adaptativa
    "weight_decay": 0.001,
    "min_time_s": 0.0,
    "seq_len": 5,              # Tamaño de la ventana deslizante (5 cuadros consecutivos)
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "time_scale": 30.0,
}

### 3. Partición de Secuencias y Carga de Datasets (Sin Data Leakage)

Para garantizar la validez científica del experimento, dividimos los datos **estrictamente por secuencias físicas de contacto** (sequence_id) antes de generar las ventanas móviles.

In [4]:
metadata_path = "../processed_data/metadata_train.csv"
df = pd.read_csv(metadata_path)

# Extraer y mezclar los IDs de secuencia únicos
unique_seqs = df["sequence_id"].dropna().unique()
np.random.default_rng(42).shuffle(unique_seqs)

# Partición 80% entrenamiento y 20% validación
split_idx = int(len(unique_seqs) * 0.8)
train_seq_ids = unique_seqs[:split_idx]
val_seq_ids = unique_seqs[split_idx:]

# Inicializar los datasets pasando los filtros de secuencia
train_ds = ThermalSequenceDataset(
    metadata_csv=metadata_path,
    is_train=True,
    min_time_s=CONFIG["min_time_s"],
    seq_len=CONFIG["seq_len"],
    sequence_ids=train_seq_ids
)
val_ds = ThermalSequenceDataset(
    metadata_csv=metadata_path,
    is_train=False,
    min_time_s=CONFIG["min_time_s"],
    seq_len=CONFIG["seq_len"],
    sequence_ids=val_seq_ids
)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], pin_memory=True)
train_eval_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], pin_memory=True)

print(f"Secuencias de entrenamiento: {len(train_seq_ids)} | Ventanas móviles: {len(train_ds)}")
print(f"Secuencias de validación: {len(val_seq_ids)} | Ventanas móviles: {len(val_ds)}")

Secuencias de entrenamiento: 24 | Ventanas móviles: 1015
Secuencias de validación: 7 | Ventanas móviles: 301


C:\Users\esteb\AppData\Local\Temp\ipykernel_30124\2138242406.py:6: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.default_rng(42).shuffle(unique_seqs)


### 4. Inicialización del Modelo, Pérdida y Optimización

In [5]:
dev = CONFIG["device"]
model = ThermalCNNLSTM().to(dev)

# Al predecir segundos reales en la salida del regresor, la pérdida
# aplica la raíz cuadrada dividiendo internamente por la escala (30.0)
crit = SqrtScaledMSELoss(scale=CONFIG["time_scale"])

opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[25, 50, 70], gamma=0.8)

print(f"Modelo CNN-LSTM inicializado con {sum(p.numel() for p in model.parameters()):,} parámetros")

Modelo CNN-LSTM inicializado con 1,445,805 parámetros


### 5. Ciclo de Entrenamiento e Impresión de Métricas Unificadas

In [6]:
best_mae = float("inf")
no_imp = 0

for ep in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss, n = 0.0, 0
    for x_seq, y in train_loader:
        x_seq, y = x_seq.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model(x_seq), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_seq.size(0)
        n += x_seq.size(0)

    scheduler.step()

    # Evaluar métricas unificadas en segundos reales desescalados
    val_m = eval_cnn_lstm_metrics(model, val_loader, dev)
    train_m = eval_cnn_lstm_metrics(model, train_eval_loader, dev)

    v_mae = val_m["mae"]
    is_best = v_mae < best_mae - CONFIG["min_delta"]
    if is_best: 
        best_mae, no_imp = v_mae, 0
        torch.save(model.state_dict(), "../CNNLSTM_best.pt")
    else: 
        no_imp += 1
    
    print(f"Ep {ep:03d} | Loss: {train_loss/n:.4f} | "
          f"TrMAE: {train_m['mae']:5.2f}s | ValMAE: {v_mae:5.2f}s | "
          f"RMSE: {val_m['rmse']:5.2f}s | R2: {val_m['r2']:.4f} | MAPE: {val_m['mape']:5.2f}% | "
          f"Acc60: {val_m['acc60']:.2f}% | Acc120: {val_m['acc120']:.2f}% {'*' if is_best else ''}")
    
    if no_imp >= CONFIG["patience"]:
        print(f"Early stop alcanzado. Mejor Val MAE: {best_mae:.2f}s")
        break

Ep 001 | Loss: 4.8991 | TrMAE: 193.96s | ValMAE: 199.10s | RMSE: 255.87s | R2: -1.5307 | MAPE: 78.90% | Acc60: 25.58% | Acc120: 39.53% *
Ep 002 | Loss: 2.9516 | TrMAE: 168.38s | ValMAE: 173.40s | RMSE: 232.51s | R2: -1.0896 | MAPE: 67.65% | Acc60: 32.56% | Acc120: 46.51% *
Ep 003 | Loss: 1.9352 | TrMAE: 148.10s | ValMAE: 152.78s | RMSE: 207.85s | R2: -0.6699 | MAPE: 73.51% | Acc60: 35.22% | Acc120: 53.49% *
Ep 004 | Loss: 1.2839 | TrMAE: 120.43s | ValMAE: 124.72s | RMSE: 182.05s | R2: -0.2811 | MAPE: 51.11% | Acc60: 42.86% | Acc120: 62.79% *
Ep 005 | Loss: 0.8531 | TrMAE: 101.91s | ValMAE: 103.97s | RMSE: 159.36s | R2: 0.0183 | MAPE: 36.84% | Acc60: 56.15% | Acc120: 69.77% *
Ep 006 | Loss: 0.6161 | TrMAE: 82.84s | ValMAE: 86.00s | RMSE: 139.68s | R2: 0.2459 | MAPE: 29.89% | Acc60: 62.46% | Acc120: 76.74% *
Ep 007 | Loss: 0.4584 | TrMAE: 78.96s | ValMAE: 81.44s | RMSE: 127.19s | R2: 0.3747 | MAPE: 31.24% | Acc60: 65.45% | Acc120: 76.08% *
Ep 008 | Loss: 0.3676 | TrMAE: 79.98s | ValMAE: 